In [ ]:
# Mounting google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

rej = pd.read_csv('/content/drive/MyDrive/rejected_loans_with_predictions.csv')
rej.head()


,loan_title,default_prob,risk_category
0,Wedding Covered but No Honeymoon,0.115299,Medium
1,Consolidating Debt,0.089294,Low
2,Want to consolidate my debt,0.123505,Medium
3,waksman,0.254053,High
4,mdrigo,0.148081,Medium


Filter Fundable Loans

In [ ]:
fundable = rej[rej['risk_category'].isin(['Low', 'Medium', 'High'])].copy()
fundable = fundable.sample(2000, random_state=42)  # keep it light for demo

fundable['loan_id'] = range(len(fundable))
fundable.head()


,loan_title,default_prob,risk_category,loan_id
994362,debt_consolidation,0.204948,High,0
736081,home_improvement,0.216556,High,1
68565,Nursing School Tuition,0.169386,Medium,2
16181,personal,0.218461,High,3
827481,credit_card,0.203300,High,4


Create Synthetic Lenders

In [ ]:
np.random.seed(42)

lenders = pd.DataFrame({
    'lender_id': range(300),   # 300 lenders
    'risk_appetite': np.random.choice(
        ['Conservative', 'Moderate', 'Aggressive'],
        size=300,
        p=[0.3, 0.4, 0.3]
    ),
    'capital': np.random.randint(50_000, 10_00_000, size=300)
})

lenders.head()

,lender_id,risk_appetite,capital
0,0,Moderate,771772
1,1,Aggressive,239407
2,2,Aggressive,499395
3,3,Moderate,999597
4,4,Conservative,512894


Risk Appetite Rules (CORE LOGIC)

In [ ]:
np.random.seed(42)

risk_amount_map = {
    'Low': (500_000, 1_000_000),
    'Medium': (200_000, 500_000),
    'High': (50_000, 200_000)
}

fundable['loan_amount'] = fundable['risk_category'].apply(
    lambda r: np.random.randint(*risk_amount_map[r])
)

fundable['remaining_amount'] = fundable['loan_amount']
fundable['lender_count'] = 0

fundable.head()

,loan_title,default_prob,risk_category,loan_id,loan_amount,remaining_amount,lender_count
994362,debt_consolidation,0.204948,High,0,171958,171958,0
736081,home_improvement,0.216556,High,1,196867,196867,0
68565,Nursing School Tuition,0.169386,Medium,2,331932,331932,0
16181,personal,0.218461,High,3,153694,153694,0
827481,credit_card,0.203300,High,4,169879,169879,0


In [ ]:
risk_map = {
    'Conservative': ['Low', 'Medium'],
    'Moderate': ['Medium', 'High'],
    'Aggressive': ['High']
}

In [ ]:
allocations = []

Risk Appetite Rules (CORE LOGIC)

In [ ]:
# Which lender can fund which loan risk
risk_amount_map = {
    'Low': (500000, 1000000),
    'Medium': (200000, 500000),
    'High': (50000, 200000),
    'Very High': (20000, 50000)
}

fundable_loans = rej[
    rej['risk_category'].isin(['Low', 'Medium', 'High'])
].copy()

fundable_loans['loan_amount'] = fundable_loans['risk_category'].apply(
    lambda r: np.random.randint(*risk_amount_map[r])
)

fundable_loans['remaining_amount'] = fundable_loans['loan_amount']

fundable_loans.head()

,loan_title,default_prob,risk_category,loan_amount,remaining_amount
0,Wedding Covered but No Honeymoon,0.115299,Medium,483076,483076
1,Consolidating Debt,0.089294,Low,617796,617796
2,Want to consolidate my debt,0.123505,Medium,491999,491999
3,waksman,0.254053,High,57400,57400
4,mdrigo,0.148081,Medium,373714,373714


In [ ]:
risk_map = {
    'Conservative': ['Low', 'Medium'],
    'Moderate': ['Medium', 'High'],
    'Aggressive': ['High', 'Very High']
}


RBI CONSTRAINTS

In [ ]:
fundable_loans = rej[
    rej['risk_category'].isin(['Low', 'Medium', 'High'])
].copy()

fundable_loans['loan_amount'] = fundable_loans['risk_category'].apply(
    lambda r: np.random.randint(*risk_amount_map[r])
)

fundable_loans['remaining_amount'] = fundable_loans['loan_amount']
fundable_loans['lender_count'] = 0


In [ ]:
MAX_PER_LOAN = 0.20        # max 20% per lender per loan
MIN_LENDERS = 5           # minimum lenders per loan


Matching Algorithm (Greedy + Safe)

In [ ]:
MAX_PER_LOAN = 0.20   # 20% cap per lender per loan
MIN_LENDERS = 5

for _, lender in lenders.iterrows():
    capital_left = lender['capital']
    allowed_risks = risk_map[lender['risk_appetite']]

    eligible_loans = fundable[
        (fundable['risk_category'].isin(allowed_risks)) &
        (fundable['remaining_amount'] > 0)
    ]

    for idx, loan in eligible_loans.iterrows():
        if capital_left <= 0:
            break

        max_allowed = MAX_PER_LOAN * loan['loan_amount']
        fund_amt = min(max_allowed, loan['remaining_amount'], capital_left)

        if fund_amt <= 0:
            continue

        allocations.append({
            'lender_id': lender['lender_id'],
            'loan_id': loan['loan_id'],
            'funded_amount': fund_amt,
            'risk_category': loan['risk_category']
        })

        capital_left -= fund_amt
        fundable.at[idx, 'remaining_amount'] -= fund_amt
        fundable.at[idx, 'lender_count'] += 1


/tmp/ipython-input-589030947.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '137566.4' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fundable.at[idx, 'remaining_amount'] -= fund_amt


Allocation Results

In [ ]:
alloc_df = pd.DataFrame(allocations)
alloc_df.head()


,lender_id,loan_id,funded_amount,risk_category
0,0,0,34391.6,High
1,0,1,39373.4,High
2,0,2,66386.4,Medium
3,0,3,30738.8,High
4,0,4,33975.8,High


RBI Compliance & Sanity Checks

In [ ]:
# Fully funded loans
fully_funded = fundable[fundable['remaining_amount'] == 0]
print("Fully funded loans:", len(fully_funded))

# RBI diversification check
violations = fundable[fundable['lender_count'] < MIN_LENDERS]
print("Loans violating min lender rule:", len(violations))

# Total capital deployed
print("Total capital deployed:", alloc_df['funded_amount'].sum())


Fully funded loans: 621
Loans violating min lender rule: 1376
Total capital deployed: 149835442.0


Save Outputs

In [ ]:
alloc_df.to_csv('/content/drive/MyDrive/lender_loan_allocations.csv', index=False)
fundable.to_csv('/content/drive/MyDrive/fundable_loans_post_matching.csv', index=False)
